# Tutorial 11, Part 2: iLQR

## 1. Setup

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as anim
import random 
import sympy as smp
import time

%matplotlib inline

from IPython.display import HTML

from part2.pendulum_plant import PendulumPlant, plot_timeseries
from part2.lqr import lqr, dlqr
from part2.utilities_iLQR import check_type
from part2.RoAutils import najafi_based_sampling

First, we set up our PendulumPlant instance:

In [5]:
mass = 0.06      # mass at the end of the pendulum [kg]
length = 0.1    # length of the rod [m]
damping = 0.0004   # viscious friction [kg m/s]
gravity = 9.81  # gravity [kg m²/s]
inertia = mass*length*length   # inertia of the pendulum [kg m²]
coulomb_fric = 0.0   # Coulomb friction coefficient [-]
torque_limit = np.inf  # torque limit of the motor, here we assume the motor can produce any torque
dt = 0.010            # time step size for iLQR (not simulations) [s]

pendulum = PendulumPlant(mass=mass,
                         length=length,
                         damping=damping,
                         gravity=gravity,
                         inertia=inertia,
                         torque_limit=torque_limit)

# 2. Iterative Linear Quadratic Regulator

With underactuated systems, sometimes we find that traditional controllers that are limited to reacting to the state of the system are unable to achieve desirable outcomes. One way of implementing a controller which can exploit the dynamics of the system to a greater extent is through the use of trajectory optimization. 

By introducing our knowledge of the system, we can calculate trajectories which can fully exploit the actuator's potential. This also allows us to assign priorities (weights) to different characteristics of the system. We could optimize, for instance, for rapid execution, accuracy, efficiency, etc. Or even balance combinations of priorities.

An iterative Linear Quadratic Regulator (iLQR) achieves this by finding the Linear Quadratic Regulator of a system according to a cost function. Then the system is simulated with the new controller, obtaining a new trajectory. The problem is then linearized around the new trajectory, the LQR algorithm is applied again, and the cycle is repeated successively until convergence.

### 2.1. The iLQR problem

The basic problem statement is as follows:

$$ \min_{u} \; \ell_f (x_N) + \sum^{N-1}_{n=1} \ell (x_n, u_n)$$
subject to
$$ x_{n} = f (x_{n-1}, u_{n-1}) \quad \forall n\in[0, N]$$

In other words, we seek to find a set of values of the control input $u$ which minimizes the the cost function. The cost function comprises two distinct terms. $\ell$ is the cost associated to any given step, except for the final step. $\ell$ is a function of the state and control at any given step $k$; $x_n$ and $u_n$, respectively. $\ell_f$ the associated to the final state, which only depends on the state of the system at the final step N.

At the same time, the state of the system at any given instant, $x_n$, other than the initial state $x_0$, is a function of the state and control input at the previous instant. The state from which we start, $x_0$ is supplied by the user.


### 2.2. iLQR calculator class

The first step in our procedure is to define a class which we will call to solve the iLQR problem. Here, we will define the steps we take to obtain the optimal trajectory, which we will later enforce with another controller.

Initially, we define the following member functions:

- **\_\_init\_\_**: Class constructor, initializes symbolic varaibles *x_sym* and *u_sym*.
- **set_start**: Allows us to set an initial state.
- **set_discrete_dynamics**: Allows us to set the dynamics of the system. The dynamics must be already in discrete form, i.e., the inputs are the previous state and control and the output the new state.
- **set_stage_cost**: Allows us to define a cost function for the duration of the motion (except for the final state).
- **set_final_cost**: Allows us to define a cost function for the final state.
- **cost_trj**: Calculates the cost associated to the provided trajectory.

In [ ]:
# The following is adapted from DFKI's iLQR code
# Large parts taken from `Russ Tedrake <https://github.com/RussTedrake/underactuated>`_.

class iLQR_Calculator():
    '''
    Class to calculate an optimal trajectory with an iterative
    linear quadratic regulator (iLQR). This implementation uses sympy.
    '''
    def __init__(self, n_x=2, n_u=1):
        """
        Class to calculate an optimal trajectory with an iterative
        linear quadratic regulator (iLQR). This implementation uses sympy.

        Parameters
        ----------
        n_x : int, default=2
            The size of the state space.
        n_u : int, default=1
            The size of the control space.
        """
        self.n_x = n_x
        self.n_u = n_u

        self.x_sym = smp.symbols("x:"+str(self.n_x))
        self.u_sym = smp.symbols("u:"+str(self.n_u))

    def set_start(self, x0):
        """
        Set the start state for the trajectory.

        Parameters
        ----------
        x0 : array-like
            the start state. Should have the shape of (n_x,)
        """
        self.x0 = x0

    def set_discrete_dynamics(self, dynamics_func):
        '''
        Sets the dynamics function for the iLQR calculation.

        Parameters
        ----------
        danamics_func : function
            dynamics_func should be a function with inputs (x, u) and output xd
        '''
        self.discrete_dynamics = dynamics_func

    def set_stage_cost(self, stage_cost_func):
        '''
        Set the stage cost (running cost) for the ilqr optimization.

        Parameters
        ----------
        stage_cost_func : function
            stage_cost_func should be a function with inputs (x, u)
            and output cost
        '''
        self.stage_cost = stage_cost_func

    def set_final_cost(self, final_cost_func):
        '''
        Set the final cost for the ilqr optimization.

        Parameters
        ----------
        final_cost_func : function
            final_cost_func should be a function with inputs x
            and output cost
        '''
        self.final_cost = final_cost_func

    def cost_trj(self, x_trj, u_trj):
        total = 0.0
        ln = 0.0
        N = x_trj.shape[0]
        for i in range(len(x_trj)-1):
            ln = ln + self.stage_cost(x_trj[i, :], u_trj[i, :]) / N
        lf = self.final_cost(x_trj[i+1, :])
        total = ln + lf
        return total